# Image Captioning Demo: DeiT + LFE + MSG + BiLSTM+MDSA-C

This notebook demonstrates the complete image captioning pipeline with the new architecture.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('../src')

import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

from models import CaptioningModel, LFE, MSG, MDSA_C, BiLSTMDecoder
from utils import Tokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Test Individual Components

In [ ]:
# Test LFE (Local Feature Enhancer)
print('Testing LFE...')
lfe = LFE(in_channels=768, hidden_dim=768, num_layers=2)
x = torch.randn(1, 768, 14, 14)
out = lfe(x)
print(f'  Input shape: {x.shape}')
print(f'  Output shape: {out.shape}')
print(f'  ✓ LFE working correctly\n')

In [ ]:
# Test MSG (Multi-Scale Gating)
print('Testing MSG...')
msg = MSG(in_dim=768, scales=[1, 2, 4], gating=True)
out = msg(x)
print(f'  Input shape: {x.shape}')
print(f'  Output shape: {out.shape}')
print(f'  ✓ MSG working correctly\n')

In [ ]:
# Test MDSA-C (Multi-Head Dual-Stage Attention with Context)
print('Testing MDSA-C...')
mdsa = MDSA_C(embed_dim=512, num_heads=8, dropout=0.1)
x_seq = torch.randn(1, 196, 512)
out, attn = mdsa(x_seq)
print(f'  Input shape: {x_seq.shape}')
print(f'  Output shape: {out.shape}')
print(f'  Attention shape: {attn.shape}')
print(f'  ✓ MDSA-C working correctly\n')

In [ ]:
# Test BiLSTM Decoder
print('Testing BiLSTMDecoder...')
decoder = BiLSTMDecoder(
    vocab_size=1000,
    embed_dim=256,
    hidden_dim=512,
    visual_dim=768,
    num_layers=1,
    dropout=0.1,
    num_heads=8,
    pad_idx=0
)
visual_features = torch.randn(1, 196, 768)
captions = torch.randint(0, 1000, (1, 20))
outputs = decoder(visual_features, captions, teacher_forcing_ratio=1.0)
print(f'  Visual features: {visual_features.shape}')
print(f'  Captions: {captions.shape}')
print(f'  Outputs: {outputs.shape}')
print(f'  ✓ BiLSTMDecoder working correctly\n')

## 3. Test Complete Model

In [ ]:
# Create complete captioning model
print('Creating CaptioningModel...')
model = CaptioningModel(
    vocab_size=1000,
    embed_dim=256,
    hidden_dim=512,
    visual_dim=768,
    num_decoder_layers=1,
    num_heads=8,
    dropout=0.1,
    deit_model_name='deit_base_patch16_224',
    pretrained=False,  # Set to True to use pretrained DeiT weights
    use_bilstm_encoder=False,
    pad_idx=0
).to(device)

print(f'  Model created successfully')
print(f'  Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# Test forward pass
print('Testing forward pass...')
images = torch.randn(2, 3, 224, 224).to(device)
captions = torch.randint(0, 1000, (2, 20)).to(device)

with torch.no_grad():
    outputs = model(images, captions, teacher_forcing_ratio=1.0)

print(f'  Input images: {images.shape}')
print(f'  Input captions: {captions.shape}')
print(f'  Output logits: {outputs.shape}')
print(f'  ✓ Forward pass working correctly\n')

In [ ]:
# Test caption generation
print('Testing caption generation...')
with torch.no_grad():
    # Greedy decoding
    generated_greedy = model.generate(
        images,
        max_len=20,
        sos_idx=1,
        eos_idx=2,
        beam_size=1
    )
    print(f'  Greedy decoding: {generated_greedy.shape}')
    print(f'  Example caption (token IDs): {generated_greedy[0].tolist()[:10]}...')

print(f'  ✓ Caption generation working correctly\n')

In [ ]:
# Test feature extraction
print('Testing feature extraction...')
with torch.no_grad():
    features = model.extract_features(images)

print(f'  Extracted features: {features.shape}')
print(f'  ✓ Feature extraction working correctly\n')

## 4. Generate Caption for a Real Image (Example)

This section shows how to generate captions for real images once the model is trained.

In [ ]:
# Define image transforms (same as used for DeiT)
def get_image_transform():
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

# Example: Load and preprocess image
# image_path = 'path/to/your/image.jpg'
# image = Image.open(image_path).convert('RGB')
# transform = get_image_transform()
# image_tensor = transform(image).unsqueeze(0).to(device)

# Generate caption
# model.eval()
# with torch.no_grad():
#     caption_ids = model.generate(
#         image_tensor,
#         max_len=50,
#         sos_idx=1,  # Your actual <sos> index
#         eos_idx=2,  # Your actual <eos> index
#         beam_size=3
#     )

# Decode caption (requires trained tokenizer)
# tokenizer = Tokenizer.load('path/to/vocab.pkl')
# caption_tokens = [tokenizer.vocab.idx_2_str[idx.item()] 
#                   for idx in caption_ids[0] 
#                   if idx.item() in tokenizer.vocab.idx_2_str]
# caption = ' '.join(caption_tokens)
# print(f'Caption: {caption}')

print('Example code for generating captions from real images (uncomment to use)')

## 5. Training Example

Example of how to set up training loop.

In [ ]:
# Training configuration
config = {
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'batch_size': 32,
    'epochs': 30,
    'teacher_forcing_ratio': 1.0,  # Start with full teacher forcing
    'warmup_steps': 1000
}

# Setup optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay']
)

# Setup loss
criterion = nn.CrossEntropyLoss(ignore_index=0)  # 0 is pad_idx

print('Training configuration:')
for key, value in config.items():
    print(f'  {key}: {value}')
print('\nOptimizer and loss function ready')
print('Use train_new.py for complete training pipeline')

## Summary

This notebook demonstrated:
1. Individual component testing (LFE, MSG, MDSA-C, BiLSTMDecoder)
2. Complete model creation and testing
3. Forward pass and caption generation
4. Feature extraction
5. Training setup example

For complete training, use the provided `train_new.py` script.
For feature extraction, use `extract_deit_features.py`.